# Combined Equation Model Exploration with Operator Composition

This notebook allows you to:
- Explore your DISCO model trained on combined physics equations (EULER, HEAT, DISP)
- Test operator composition methods (Greedy, Random, Exhaustive)
- Compare performance on out-of-distribution data
- Analyze operator usage patterns across physics types

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import h5py
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from torch.utils.data import DataLoader
import random
from itertools import permutations
import math
import time

# Add project root to path
sys.path.append('/mnt/home/lserrano/disco-ball')
sys.path.append('/mnt/home/lserrano/disco-ball/tests/neural-operator-splitting')

from train.train_combined_vqvae import DISCOLitModule# HDF5TemporalDataset
#from train.train_combined import DISCOLitModule# HDF5TemporalDataset
from src.utils.database import RelativeL2
from src.operators.disco_vqvae import DISCOHouse
from operator_utils import sequential_operator_composition, strang_splitting_composition
from einops import rearrange

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
import torch.nn.functional as F

## Configuration

In [ ]:
class HDF5TemporalDataset(torch.utils.data.Dataset):
    """Dataset for loading pre-computed trajectory data from HDF5 files"""
    
    def __init__(self, hdf5_files, input_frames=16, output_frames=16, 
                 sub_x=1, sub_t=1, split='train'):
        """
        Args:
            hdf5_files: List of HDF5 file paths to load data from
            input_frames: Number of input time frames
            output_frames: Number of output time frames  
            sub_x: Spatial subsampling factor
            sub_t: Temporal subsampling factor
            split: Dataset split ('train', 'val', 'test')
        """
        self.hdf5_files = hdf5_files if isinstance(hdf5_files, list) else [hdf5_files]
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        
        # Build file index for efficient access
        print("Building file index...")
        start_time = time.time()
        self.total_samples = self._build_file_index()
        index_time = time.time() - start_time
        print(f"Dataset length calculation took {index_time:.2f}s for {self.total_samples} samples")
        
        # Track loading times for performance assessment
        self.loading_times = []
        
    def _build_file_index(self):
        """Pre-compute file offsets for efficient __len__ and __getitem__"""
        self.file_offsets = []
        total_samples = 0
        
        for file_path in self.hdf5_files:
            if not os.path.exists(file_path):
                print(f"Warning: HDF5 file not found: {file_path}")
                continue
                
            try:
                with h5py.File(file_path, 'r') as f:
                    # Try different group names based on split
                    data_group = None
                    dataset_path = None
                    
                    # Check for split-specific groups first, then fall back to 'train'
                    possible_groups = [self.split, 'train', 'valid', 'test']
                    for group_name in possible_groups:
                        if group_name in f and 'pde_250-256' in f[group_name]:
                            data_group = group_name
                            dataset_path = f'{group_name}/pde_250-256'
                            break
                    
                    if data_group is None:
                        print(f"Warning: No valid dataset structure found in {file_path}. Checked groups: {possible_groups}")
                        continue
                        
                    n_samples = f[dataset_path].shape[0]
                    n_timesteps = f[dataset_path].shape[1]
                    
                    # Verify we have enough timesteps for input + output frames
                    min_timesteps_needed = (self.input_frames + self.output_frames) * self.sub_t
                    if n_timesteps < min_timesteps_needed:
                        print(f"Warning: Not enough timesteps in {file_path}. "
                              f"Need {min_timesteps_needed}, got {n_timesteps}")
                        continue
                    
                    self.file_offsets.append((file_path, total_samples, n_samples, dataset_path))
                    total_samples += n_samples
                    print(f"Added {n_samples} samples from {file_path} (using {dataset_path})")
                    
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                continue
                
        if total_samples == 0:
            raise ValueError("No valid samples found in any HDF5 files!")
            
        return total_samples
    
    def _get_file_and_local_idx(self, idx):
        """Convert global index to file path and local index"""
        for file_path, offset, n_samples, dataset_path in self.file_offsets:
            if idx < offset + n_samples:
                local_idx = idx - offset
                return file_path, local_idx, dataset_path
        raise IndexError(f"Index {idx} out of range for dataset size {self.total_samples}")
    
    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        start_time = time.time()
        
        # Find which file and local index
        file_path, local_idx, dataset_path = self._get_file_and_local_idx(idx)
        
        try:
            with h5py.File(file_path, 'r') as f:
                # Get the group containing the data
                group_name = dataset_path.split('/')[0]
                
                # Load trajectory data - shape: (n_timesteps, n_spatial)
                trajectory = f[dataset_path][local_idx]
                
                # Load PDE parameters (alpha, beta, gamma) for this sample
                alpha = f[group_name]['alpha'][local_idx]
                beta = f[group_name]['beta'][local_idx]
                gamma = f[group_name]['gamma'][local_idx]
                
                # Sample temporal window randomly
                total_frames_needed = self.input_frames + self.output_frames
                max_start = (trajectory.shape[0] // self.sub_t) - total_frames_needed
                if max_start <= 0:
                    # If not enough frames, use what we have
                    start_idx = 0
                    available_frames = trajectory.shape[0] // self.sub_t
                    actual_input_frames = min(self.input_frames, available_frames // 2)
                    actual_output_frames = available_frames - actual_input_frames
                else:
                    start_idx = np.random.randint(0, max_start + 1)
                    actual_input_frames = self.input_frames
                    actual_output_frames = self.output_frames
                
                # Apply temporal subsampling and extract sequences
                #start_t = 0
                start_t = 0 #start_index # 50
                input_end_t = start_t + actual_input_frames * self.sub_t
                output_end_t = input_end_t + actual_output_frames * self.sub_t
                
                input_seq = trajectory[start_t:input_end_t:self.sub_t, ::self.sub_x]
                output_seq = trajectory[input_end_t:output_end_t:self.sub_t, ::self.sub_x]
                
                # Add channel dimension and convert to torch tensors
                # Expected format: (time, channels, spatial)
                input_tensor = torch.from_numpy(input_seq).unsqueeze(-2).float()
                output_tensor = torch.from_numpy(output_seq).unsqueeze(-2).float()
                
                # Track loading time
                loading_time = time.time() - start_time
                if len(self.loading_times) < 1000:  # Collect first 1000 samples
                    self.loading_times.append(loading_time)
                
                return {
                    'input': input_tensor, 
                    'target': output_tensor,
                    'alpha': float(alpha),
                    'beta': float(beta),
                    'gamma': float(gamma),
                    'index': idx
                }
                
        except Exception as e:
            print(f"Error loading sample {idx} from {file_path}: {e}")
            # Return dummy data to avoid training crash
            dummy_input = torch.zeros(self.input_frames, 1, 256 // self.sub_x)
            dummy_output = torch.zeros(self.output_frames, 1, 256 // self.sub_x)
            return {'input': dummy_input, 'target': dummy_output}
    
    def get_loading_stats(self):
        """Return loading performance statistics"""
        if not self.loading_times:
            return {}
        
        return {
            'avg_loading_time': np.mean(self.loading_times),
            'min_loading_time': np.min(self.loading_times),
            'max_loading_time': np.max(self.loading_times),
            'samples_per_second': 1.0 / np.mean(self.loading_times),
            'total_samples_timed': len(self.loading_times)
        }

In [ ]:
# Model configuration - UPDATE THESE PATHS

#simvq
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t8_steps1_initFalse_bs32_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250908_204555"

#simvq temporal consistency
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs32_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250909_224012"

# in-context 
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs32_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250909_134011"

#simvq small
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs32_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250908_204356"

#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs64_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250913_004759"

# basic disco with temporal conditioning on longer horizons
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes32_subx1_subt1_20250913_141857"

#vqvae 2
run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs128_lr0.0005_hdf5_noise0_inframes16_outframes2_subx1_subt1_20250913_161113"

MODEL_CHECKPOINT_PATH = f"/mnt/home/lserrano/disco-ball/outputs/{run_name}/last.ckpt"
DATA_DIR = "/mnt/home/lserrano/disco-ball/datasets/combined_equation/"

VALIDATION_FILES = {
    'EULER': f"{DATA_DIR}E_EULER_valid.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_valid.h5",
    'DISP': f"{DATA_DIR}E_DISP_valid.h5"
}

TEST_FILES = {
    'EULER': f"{DATA_DIR}E_EULER_test.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_test.h5",
    'DISP': f"{DATA_DIR}E_DISP_test.h5"
}

# Training files for operator encoding
TRAINING_FILES = {
    #'EULER': f"{DATA_DIR}E_EULER_train_8192.h5",
    #'HEAT': f"{DATA_DIR}E_HEAT_train_8192.h5",
    #'DISP': f"{DATA_DIR}E_DISP_train_8192.h5"
    'EULER': f"{DATA_DIR}E_EULER_train_envsize64.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_train_envsize64.h5",
    'DISP': f"{DATA_DIR}E_DISP_train_envsize64.h5"
}

# Test configuration
BATCH_SIZE = 64
N_INPUT_FRAMES = 16
N_OUTPUT_FRAMES = 100
SUB_X = 1
SUB_T = 1 #1

# Operator composition configuration
OPERATOR_CONFIG = {
    'num_operators': 64,  # Number of operators to encode
    'n_trajectories_per_operator': 1,  # Trajectories per operator (anti-forgetting)
    'max_operators': 5,  # Maximum operators in composition
    'min_improvement_threshold': 5.0,  # Minimum improvement % to add operator
    'n_input_frames': N_INPUT_FRAMES,
    'n_output_frames': N_OUTPUT_FRAMES
}

print("Configuration:")
print(f"  Model path: {MODEL_CHECKPOINT_PATH}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Operator config: {OPERATOR_CONFIG}")

print("\nData files:")
for name, path in VALIDATION_FILES.items():
    exists = "✓" if os.path.exists(path) else "✗ (missing)"
    print(f"  Validation {name}: {exists}")

for name, path in TRAINING_FILES.items():
    exists = "✓" if os.path.exists(path) else "✗ (missing)"
    print(f"  Training {name}: {exists}")

## Load Model and Data

In [ ]:
def load_model_from_checkpoint(checkpoint_path):
    """Load DISCO model from Lightning checkpoint"""
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        print("Please update MODEL_CHECKPOINT_PATH with your actual model path")
        return None, None
    
    try:
        lit_model = DISCOLitModule.load_from_checkpoint(checkpoint_path, map_location=device)
        lit_model.eval()
        
        model = lit_model.model.to(device)
        model.eval()
        
        print(f"Model loaded successfully from {checkpoint_path}")
        print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
        
        return model, lit_model
        
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

# Load the model
model, lit_model = load_model_from_checkpoint(MODEL_CHECKPOINT_PATH)
relative_l2_error = RelativeL2()

if model is None:
    print("\nTo use this notebook, you need to:")
    print("1. Train a model using train_combined.py")
    print("2. Update MODEL_CHECKPOINT_PATH above with your checkpoint path")

In [ ]:
def create_dataset_for_equation(equation_type, split='val', files_dict=None):
    """Create dataset for a specific equation type"""
    files_dict = files_dict or VALIDATION_FILES
    
    if equation_type not in files_dict:
        print(f"Equation type {equation_type} not found in files dict")
        return None, None
        
    file_path = files_dict[equation_type]
    
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None, None
    
    dataset = HDF5TemporalDataset(
        hdf5_files=[file_path],
        input_frames=N_INPUT_FRAMES,
        output_frames=N_OUTPUT_FRAMES,
        sub_x=SUB_X,
        sub_t=SUB_T,
        split=split
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    print(f"{equation_type} {split} dataset: {len(dataset)} samples")
    return dataloader, dataset

# Create validation datasets
val_dataloaders = {}
val_datasets = {}

for eq_type in VALIDATION_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'val')
    if loader is not None:
        val_dataloaders[eq_type] = loader
        val_datasets[eq_type] = ds

print(f"\nLoaded {len(val_dataloaders)} validation datasets")


test_dataloaders = {}
test_datasets = {}

for eq_type in TEST_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'test', files_dict=TEST_FILES)
    if loader is not None:
        test_dataloaders[eq_type] = loader
        test_datasets[eq_type] = ds


train_dataloaders = {}
train_datasets = {}

for eq_type in TRAINING_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'train', files_dict=TRAINING_FILES)
    if loader is not None:
        train_dataloaders[eq_type] = loader
        train_datasets[eq_type] = ds

In [ ]:
eq_type="EULER"
num_integration_steps=1
N_OUTPUT_FRAMES=100

total_error = 0
test_size = 0
all_theta_latent = []
all_alpha = []
all_beta = []
all_gamma = []
all_errors = []

for batch in tqdm(val_dataloaders[eq_type]):
    inp, target = batch["input"], batch["target"]
    alpha, beta, gamma = batch['alpha'], batch['beta'], batch['gamma']
    
    all_alpha.append(alpha)
    all_beta.append(beta)
    all_gamma.append(gamma)
    
    inp = inp.squeeze(1)
    target = target.squeeze(1)
            
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
        
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)
    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels, use_vq=True)
        #theta_latent = lit_model.codes[batch['index']]
        #theta_latent = lit_model.codes[[random.randint(0, 1023) for j in range(B)]]
        #theta_latent = lit_model.codes[]
        all_theta_latent.append(theta_latent.cpu())
        theta = model.decode_theta(theta_latent, dim)
        n_output_frames = N_OUTPUT_FRAMES
        #pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames,)# integration_time=4/250*n_output_frames, dt=4/250, predict_normed=False, metadata=metadata)
        #print('inp', inp[:, -1].shape, 'theta', theta.shape)
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames, integration_time=4/250, dt=4/250)#dt=4/250, predict_normed=False, metadata=metadata)
        #print('pred',pred.shape, target.shape)

    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    
    # new
    sample_rollout_error = relative_l2_error(pred, target[:, :n_output_frames], ).item()
    x = rearrange(pred.clone(), "b ... -> b (...)")
    y = rearrange(target[:, :n_output_frames].clone(), "b ... -> b (...)")
    diff_norms = torch.linalg.norm(x - y, ord=2, dim=-1)
    y_norms = torch.linalg.norm(y, ord=2, dim=-1)
    sample_rollout_error = diff_norms / y_norms

    all_errors.append(sample_rollout_error)
    
    total_error+=rollout_error*n_sample
    test_size+=n_sample
    
all_errors = torch.cat(all_errors)
print('test error', total_error/test_size)

In [ ]:
idx=50
for t in range(N_OUTPUT_FRAMES):
    plt.plot(pred[idx].squeeze().cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze().cpu().detach()[t])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# 1. OOD Dataset Loader
def load_ood_dataset(ood_type, split='train'):
  """Load specific OOD dataset"""
  file_path = f"/mnt/home/lserrano/disco-ball/datasets/combined_equation/ood/{ood_type}_{split}_512.h5"
  dataset = HDF5TemporalDataset([file_path], N_INPUT_FRAMES, N_OUTPUT_FRAMES, SUB_X, SUB_T, split)
  dataloader = DataLoader(dataset, batch_size=128, shuffle=True)
  return dataloader

# 2. Simple Direct Prediction Test
def test_direct_prediction(model, dataloader, ood_name):
  """Test standard model prediction"""
  total_error = 0
  samples = 0
  prediction = []
  for batch in dataloader:
      inp, target = batch["input"].to(device), batch["target"].to(device)
      state_labels = torch.tensor([0], device=device)

      with torch.no_grad():
          pred, _ = model(inp, state_labels, n_future_steps=target.shape[1])#, integration_time=target.shape[1])
          prediction.append(pred.cpu())
          error = relative_l2_error(pred, target).item()
          total_error += error * inp.shape[0]
          samples += inp.shape[0]

  avg_error = total_error / samples
  print(f"{ood_name} Direct Prediction Error: {avg_error:.6f}")
  return avg_error, torch.cat(prediction)

# 3. Simple Theta Prediction Test  
def test_theta_prediction(model, dataloader, ood_name):
  """Test theta-based prediction"""
  total_error = 0
  samples = 0
  prediction = []
  for batch in dataloader:
      inp, target = batch["input"].to(device), batch["target"].to(device)
      state_labels = torch.tensor([0], device=device)

      with torch.no_grad():
          # Extract theta from encoder
          theta_latent, metadata = model.encode_theta_latent(inp, state_labels)
          theta = model.decode_theta(theta_latent, dim=1)

          # Predict using theta
          pred, _ = model.solve_ode(inp[:, -1], theta, state_labels, dim=1,
                                  n_future_steps=target.shape[1], )#integration_time=target.shape[1], dt=1,
                                   #predict_normed=False, metadata={})
          prediction.append(pred.cpu())
          error = relative_l2_error(pred, target).item()
          total_error += error * inp.shape[0]
          samples += inp.shape[0]

  avg_error = total_error / samples
  print(f"{ood_name} Theta Prediction Error: {avg_error:.6f}")
  return avg_error, torch.cat(prediction)

In [ ]:
# 6. Test All OOD Datasets
def test_all_ood_datasets(ood_types = ['E_BG'], epochs=100, lr=0.01, refinement_factor=1, num_operators=2):
  """Test all methods on all OOD datasets"""
  #ood_types = ['E_ALL', 'E_BG', 'E_ED', 'E_HE']
  #ood_types = ['E_HE']
  
  results = {}
  

  for ood_type in ood_types:
      print(f"\n=== Testing {ood_type} ===")

  
      dataloader = load_ood_dataset(ood_type)
      target = []
      for batch in dataloader:
          target.append(batch['target'])
          
      target = batch['target']
      input = batch['input']
      
      theta_latent_tuple, pred = gradient_selection_multi_operator(model, theta_latent_operators, input, target, num_operators=num_operators,
                              min_improvement_threshold=5.0, lr=lr, epochs=epochs, refinement_factor=refinement_factor)

      results[ood_type] = {
          #'direct_error': direct_error,
          #'theta_error': theta_error,
          #'greedy_composition': greedy_comp,
          #'random_composition': random_comp,
          #'pred_direct':pred_direct,
          #"pred_random":pred_random,
          #"target_random":target_random,
          #'pred_theta':pred_theta,
          "theta_latent": theta_latent_tuple,
          'pred_grad':pred,
          'target_grad': target,
          #'pred_nearest':pred_nearest,
      }

      #except Exception as e:
      #    print(f"Error testing {ood_type}: {e}")
      #    results[ood_type] = {'error': str(e)}

  return results

In [ ]:
# Operator Encoding

def encode_operators_from_training_data(model, train_files, num_operators=20, 
                                       n_trajectories_per_operator=4):
    """Encode operators from training trajectories."""
    if model is None:
        print("Model not loaded")
        return None
    
    print(f"Encoding {num_operators} operators from training data...")
    
    # Create training datasets
    train_datasets = {}
    for eq_type, file_path in train_files.items():
        if os.path.exists(file_path):
            dataset = HDF5TemporalDataset(
                hdf5_files=[file_path],
                input_frames=N_INPUT_FRAMES,
                output_frames=N_OUTPUT_FRAMES,
                sub_x=1, sub_t=1, split='train'
            )
            train_datasets[eq_type] = dataset
            print(f"  {eq_type}: {len(dataset)} training samples")
    
    if not train_datasets:
        print("No training datasets available")
        return None
    
    # Collect trajectories for encoding
    all_trajectories = []
    operator_metadata = []
    
    trajectories_per_equation = num_operators // len(train_datasets)
    remaining = num_operators % len(train_datasets)

    total_collected=0
    for eq_idx, (eq_type, dataset) in enumerate(train_datasets.items()):
        ops_from_this_eq = trajectories_per_equation + (1 if eq_idx < remaining else 0)
        
        print(f"Encoding {ops_from_this_eq} operators from {eq_type}...")
        
        dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
        
        collected = 0
        target_trajectories = ops_from_this_eq * n_trajectories_per_operator
        
        for batch in dataloader:
            if collected >= target_trajectories:
                break
                
            input_seq = batch['input']
            alpha = batch['alpha']
            beta = batch['beta']
            gamma = batch['gamma']
            
            for sample_idx in range(input_seq.shape[0]):
                if collected >= target_trajectories:
                    break
                
                trajectory = input_seq[sample_idx:sample_idx+1]
                all_trajectories.append(trajectory)
                
                # Track operator metadata
                operator_idx = collected #// n_trajectories_per_operator
                if collected % n_trajectories_per_operator == 0:
                    operator_metadata.append({
                        'operator_id': total_collected,
                        'equation_type': eq_type,
                        'trajectory_indices': [],
                        'alpha':alpha[sample_idx:sample_idx+1],
                        'beta':beta[sample_idx:sample_idx+1],
                        'gamma':gamma[sample_idx:sample_idx+1],
                    })
                
                operator_metadata[-1]['trajectory_indices'].append(len(all_trajectories) - 1)
                collected += 1
                total_collected +=1
    
    print(f"Collected {len(all_trajectories)} trajectories for {len(operator_metadata)} operators")
    
    # Encode all trajectories
    all_theta_latent = []
    all_theta = []
    
    state_labels = torch.tensor([0], device=device)
    encoding_batch_size = 32
    
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(all_trajectories), encoding_batch_size), desc="Encoding"):
            batch_trajectories = all_trajectories[i:i+encoding_batch_size]
            batch_input = torch.cat(batch_trajectories, dim=0).to(device)
            
            theta_latent_batch, _ = model.encode_theta_latent(batch_input, state_labels, use_vq=True)
            theta_batch = model.decode_theta(theta_latent_batch, dim=1)
            
            all_theta_latent.append(theta_latent_batch.cpu())
            all_theta.append(theta_batch.cpu())
    
    all_theta_latent = torch.cat(all_theta_latent, dim=0)
    all_theta = torch.cat(all_theta, dim=0)
    
    # Average parameters for each operator
    num_unique_operators = len(operator_metadata)
    theta_operators = torch.zeros(num_unique_operators, all_theta.shape[1])
    theta_latent_operators = torch.zeros(num_unique_operators, all_theta_latent.shape[1])
    
    for op_idx, op_meta in enumerate(operator_metadata):
        traj_indices = op_meta['trajectory_indices']
        theta_operators[op_idx] = all_theta[traj_indices].mean(dim=0)
        theta_latent_operators[op_idx] = all_theta_latent[traj_indices].mean(dim=0)
    
    print(f"\nEncoded {num_unique_operators} operators:")
    print(f"  Theta shape: {theta_operators.shape}")
    print(f"  Theta latent shape: {theta_latent_operators.shape}")
    
    # Print distribution
    eq_counts = {}
    for op_meta in operator_metadata:
        eq_type = op_meta['equation_type']
        eq_counts[eq_type] = eq_counts.get(eq_type, 0) + 1
    
    print(f"  Distribution: {eq_counts}")
    
    return theta_operators, theta_latent_operators, operator_metadata

In [ ]:
def simple_splitting(model, theta_latent_1, theta_latent_2, x, nt=1, dt=4/250, refinement_factor=5, splitting_method="strang"):
    
    state_labels = torch.tensor([0], device=x.device)
    #state_labels = 0
    dim = 1
    
    small_dt = dt/refinement_factor
    small_dt_half = small_dt/2

    theta_1 = model.decode_theta(theta_latent_1, dim)
    theta_2 = model.decode_theta(theta_latent_2, dim)

    pred = x
    trajectory_pred = []
    for t_idx in range(nt):
        for _ in range(refinement_factor):
            if splitting_method == 'strang':
                pred, metadata = model.solve_ode(pred, theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt_half, dt=small_dt_half)
                pred, metadata = model.solve_ode(pred[:, -1], theta_2, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred, metadata = model.solve_ode(pred[:, -1], theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt_half, dt=small_dt_half)
                pred = pred[:, -1]

            else:  # lie
                # Lie splitting: full step op1, full step op2
                pred, metadata = model.solve_ode(pred, theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred, metadata = model.solve_ode(pred[:, -1], theta_2, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred = pred[:, -1]

        # Store prediction at each time step
        trajectory_pred.append(pred)

    # Convert trajectory to numpy array: (n_t+1, batch_size, 2, n_x, n_y)
    trajectory_pred = torch.cat(trajectory_pred, axis=1)
    return trajectory_pred


In [ ]:
def gradient_selection_adam(model, theta_operators, test_input, test_target, 
                             max_operators=5, min_improvement_threshold=5.0, epochs=500, lr=0.01, refinement_factor=1):
    """Greedy operator selection."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running gradient based operator selection...")
    print(f"Testing {theta_operators.shape[0]} operators, max length: {max_operators}")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()

    theta_latent_1 = (torch.randn((test_input.shape[0], 3), device=device)*0.01).requires_grad_() #theta_latent1_.clone().detach().requires_grad_()
    theta_latent_2 = (torch.randn((test_input.shape[0], 3), device=device)*0.01).requires_grad_()#theta_latent2_.clone().detach().requires_grad_()
    
    #optimizer = torch.optim.AdamW([theta_latent_1, theta_latent_2], lr=lr, weight_decay=0, betas=(0.5, 0.5)) # 0., 0.5 works okay
    optimizer = torch.optim.AdamW([theta_latent_1, theta_latent_2], lr=lr, weight_decay=0) #betas=(0.5, 0.5)) # 0., 0.5 works okay
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, 100, T_mult=1, eta_min=1e-4, last_epoch=-1)
    
    #x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    #y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    #eta= 0.01

    for step in range(epochs):
        t = random.randint(0, test_input.shape[1]-2)
        x_val = test_input[:, t]
        y_val = test_input[:, t+1]
        
        pred = simple_splitting(model, theta_latent_1, theta_latent_2, x_val, nt=1, dt=4/250, refinement_factor=1, splitting_method="strang")
        loss = relative_l2_error(pred, y_val)

        #_, _, vqloss1 = model.quantizer(theta_latent_1.clone().unsqueeze(1))
        #_, _, vqloss2 = model.quantizer(theta_latent_2.clone().unsqueeze(1))
        
        optimizer.zero_grad()
        loss.backward()
        #(loss+0.25*(vqloss1+vqloss2)).backward()
        optimizer.step()
        scheduler.step()

        print(step, loss.item())# vqloss1.item(), vqloss2.item())
        if step%100==0:
            with torch.no_grad():
                pred_test = simple_splitting(model, theta_latent_1, theta_latent_2, test_input[:, -1], nt=N_OUTPUT_FRAMES, dt=4/250, refinement_factor=1, splitting_method="strang")
            test_error = loss_fn(pred_test, test_target).item()
            print(f"Grad selection, leading to error for {N_OUTPUT_FRAMES}: {test_error:.6f}")

    return (theta_latent_1, theta_latent_2), pred_test

In [ ]:
def multi_operator_splitting(model, theta_latents, x, nt=1, dt=4/250, refinement_factor=5, splitting_method="strang"):
    """
    Generalized operator splitting for any number of operators
    
    Args:
        model: The model with solve_ode and decode_theta methods
        theta_latents: List of theta latent vectors [theta_1, theta_2, ..., theta_k]
        x: Input tensor
        nt: Number of time steps
        dt: Time step size
        refinement_factor: Refinement factor for sub-stepping
        splitting_method: 'strang', 'lie', or 'symmetric'
    
    Returns:
        trajectory_pred: Predicted trajectory
    """
    
    state_labels = torch.tensor([0], device=x.device)
    dim = 1
    
    small_dt = dt / refinement_factor
    k_operators = len(theta_latents)
    
    # Decode all operators
    thetas = [model.decode_theta(theta_latent, dim) for theta_latent in theta_latents]
    
    pred = x
    trajectory_pred = []
    
    for t_idx in range(nt):
        for ref_step in range(refinement_factor):
            
            if splitting_method == 'strang':
                # Strang splitting for k operators: 
                # dt/2 * op_1, dt/2 * op_2, ..., dt/2 * op_{k-1}, dt * op_k, dt/2 * op_{k-1}, ..., dt/2 * op_2, dt/2 * op_1
                
                # Forward pass (first half steps for all but last operator)
                for i in range(k_operators - 1):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt/2, dt=small_dt/2)
                    pred = pred[:, -1]
                
                # Full step for last operator
                pred, metadata = model.solve_ode(pred, thetas[-1], state_labels, dim, 
                                               n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred = pred[:, -1]
                
                # Backward pass (second half steps in reverse order)
                for i in range(k_operators - 2, -1, -1):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt/2, dt=small_dt/2)
                    pred = pred[:, -1]
            
            elif splitting_method == 'lie':
                # Lie splitting: sequential full steps
                for theta in thetas:
                    pred, metadata = model.solve_ode(pred, theta, state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt, dt=small_dt)
                    pred = pred[:, -1]
            
            elif splitting_method == 'symmetric':
                # Symmetric splitting for even number of operators
                if k_operators % 2 != 0:
                    raise ValueError("Symmetric splitting requires even number of operators")
                
                # First half operators with half time step
                for i in range(k_operators // 2):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt/2, dt=small_dt/2)
                    pred = pred[:, -1]
                
                # Second half operators with full time step  
                for i in range(k_operators // 2, k_operators):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt, dt=small_dt)
                    pred = pred[:, -1]
                
                # First half operators again with half time step (reverse order)
                for i in range(k_operators // 2 - 1, -1, -1):
                    pred, metadata = model.solve_ode(pred, thetas[i], state_labels, dim, 
                                                   n_future_steps=1, integration_time=small_dt/2, dt=small_dt/2)
                    pred = pred[:, -1]
            
            else:
                raise ValueError(f"Unknown splitting method: {splitting_method}")
        
        # Store prediction at each time step
        trajectory_pred.append(pred)
    
    # Convert trajectory to tensor
    trajectory_pred = torch.cat(trajectory_pred, dim=1)
    return trajectory_pred

In [ ]:
def gradient_selection_multi_operator(model, theta_operators, test_input, test_target, 
                                     num_operators=3, min_improvement_threshold=5.0, epochs=500, 
                                     lr=0.01, refinement_factor=1, splitting_method="strang"):
    """Multi-operator gradient selection."""
    if model is None or theta_operators is None:
        return [], {}
    
    def manifold_loss(theta_params, reference_ops):
        distances = torch.cdist(theta_params, reference_ops)
        min_distances = torch.min(distances, dim=1)[0]
        min_distances = torch.relu(min_distances - 0.01)
        return min_distances.mean()
    
    print(f"Running gradient based operator selection with {num_operators} operators...")
    print(f"Testing {theta_operators.shape[0]} reference operators")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    loss_fn = RelativeL2()
    
    # Create list of theta latents for each operator
    theta_latents = []
    for i in range(num_operators):
        theta_latent = (torch.randn((test_input.shape[0], 3), device=device)*0.1).requires_grad_()
        theta_latents.append(theta_latent)
    
    # Optimizer for all theta latents
    #optimizer = torch.optim.AdamW(theta_latents, lr=lr, weight_decay=0, betas=(0.5, 0.5))
    optimizer = torch.optim.AdamW(theta_latents, lr=lr, weight_decay=1e-4, amsgrad=True, eps=1e-4)       # Larger eps for numerical stability)
    #optimizer = torch.optim.AdamW(theta_latents, lr=lr, weight_decay=0) #betas=(0., 0.5))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    for step in range(epochs):
        t = random.randint(0, test_input.shape[1] - 2)
        x_val = test_input[:, t]
        y_val = test_input[:, t + 1]
        
        # Use multi-operator splitting
        pred = multi_operator_splitting(model, theta_latents, x_val, nt=1, dt=4/250, 
                                      refinement_factor=refinement_factor, 
                                      splitting_method=splitting_method)
        
        # Main loss
        loss = relative_l2_error(pred, y_val)
        
        # Manifold penalties for all operators
        manifold_penalties = []
        total_manifold_loss = 0
        for i, theta_latent in enumerate(theta_latents):
            manifold_penalty = 0.1 * manifold_loss(theta_latent, theta_operators)
            manifold_penalties.append(manifold_penalty)
            total_manifold_loss += manifold_penalty
        
        total_loss = loss + total_manifold_loss
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        scheduler.step()
        
        # Print progress
        if step % 20 == 0:
            penalty_str = ", ".join([f"{p.item():.4f}" for p in manifold_penalties])
            print(f"Step {step}, Loss: {loss.item():.6f}, Manifold penalties: [{penalty_str}]")
        
        # Evaluation
        if step % 100 == 0:
            print(theta_latents[0][0].grad)
            print(theta_latents[1][0].grad)
            
            #grad_norms = [theta.grad.norm().item() for theta in theta_latents]
            
            #print(len(grad_norms), grad_norms)
            grad_ratio = max(grad_norms) / (min(grad_norms) + 1e-8)
            print(f"Step {step}, Grad ratio: {grad_ratio:.1e}, Loss: {total_loss:.4f}")
            
            if grad_ratio > 100: 
                print("  ⚠️  Ill-conditioned!")
            
            with torch.no_grad():
                pred_test = multi_operator_splitting(model, theta_latents, test_input[:, -1], 
                                                   nt=N_OUTPUT_FRAMES, dt=4/250, 
                                                   refinement_factor=refinement_factor,
                                                   splitting_method=splitting_method)
            test_error = loss_fn(pred_test, test_target).item()
            print(f"Test error for {N_OUTPUT_FRAMES} frames: {test_error:.6f}")
    
    return theta_latents, pred_test

In [ ]:
def gradient_selection_adam_manifold(model, theta_operators, test_input, test_target, 
                             max_operators=5, min_improvement_threshold=5.0, epochs=500, lr=0.01, refinement_factor=1):
    """Greedy operator selection."""
    if model is None or theta_operators is None:
        return [], {}

        # Find nearest reference operator for each optimized parameter
    def manifold_loss(theta_params, reference_ops):
        distances = torch.cdist(theta_params, reference_ops)
        min_distances = torch.min(distances, dim=1)[0]
        min_distances = torch.relu(min_distances-0.01)
        return min_distances.mean()
    
    print(f"Running gradient based operator selection...")
    print(f"Testing {theta_operators.shape[0]} operators, max length: {max_operators}")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()

    theta_latent_1 = (torch.randn((test_input.shape[0], 3), device=device)*0.01).requires_grad_() #theta_latent1_.clone().detach().requires_grad_()
    theta_latent_2 = (torch.randn((test_input.shape[0], 3), device=device)*0.01).requires_grad_()#theta_latent2_.clone().detach().requires_grad_()
    
    optimizer = torch.optim.AdamW([theta_latent_1, theta_latent_2], lr=lr, weight_decay=0, betas=(0.5, 0.5)) # 0., 0.5 works okay
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, 100, T_mult=1, eta_min=1e-4, last_epoch=-1)
    
    #x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    #y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    #eta= 0.01

    for step in range(epochs):
        t = random.randint(0, test_input.shape[1]-2)
        x_val = test_input[:, t]
        y_val = test_input[:, t+1]
        
        pred = simple_splitting(model, theta_latent_1, theta_latent_2, x_val, nt=1, dt=4/250, refinement_factor=1, splitting_method="strang")
        loss = relative_l2_error(pred, y_val)

        manifold_penalty_1 = 0.1 * manifold_loss(theta_latent_1, theta_operators)
        manifold_penalty_2 = 0.1 * manifold_loss(theta_latent_2, theta_operators)
        total_loss = loss + manifold_penalty_1 + manifold_penalty_2
        
        optimizer.zero_grad()
        total_loss.backward()
        #(loss+0.25*(vqloss1+vqloss2)).backward()
        optimizer.step()
        scheduler.step()

        print(step, loss.item(), manifold_penalty_1, manifold_penalty_2)# vqloss1.item(), vqloss2.item())
        if step%100==0:
            H = torch.autograd.functional.hessian(lambda: total_loss, theta_latents)
            print(f"Step {step}, Condition number: {torch.linalg.cond(H.flatten(0, -2).flatten(1, -1)):.2e}")
            
            with torch.no_grad():
                pred_test = simple_splitting(model, theta_latent_1, theta_latent_2, test_input[:, -1], nt=N_OUTPUT_FRAMES, dt=4/250, refinement_factor=1, splitting_method="strang")
            test_error = loss_fn(pred_test, test_target).item()
            print(f"Grad selection, leading to error for {N_OUTPUT_FRAMES}: {test_error:.6f}")

    return (theta_latent_1, theta_latent_2), pred_test

In [ ]:
def project_to_tangent(grad, theta, codebook):
    """Project gradient to tangent space of VQ manifold"""
    distances = torch.norm(codebook.unsqueeze(0) - theta.unsqueeze(1), dim=-1)
    nearest_codes = codebook[torch.argmin(distances, dim=1)]
    quant_dir = F.normalize(nearest_codes - theta, dim=-1)
    return grad - torch.sum(grad * quant_dir, dim=-1, keepdim=True) * quant_dir

In [ ]:
def _rieman(model, theta_operators, test_input, test_target, 
                             max_operators=5, min_improvement_threshold=5.0, epochs=500):
    """Greedy operator selection."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running gradient based operator selection...")
    print(f"Testing {theta_operators.shape[0]} operators, max length: {max_operators}")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()

    theta_latent_1 = (torch.randn((test_input.shape[0], 8), device=device)*0.01).requires_grad_() #theta_latent1_.clone().detach().requires_grad_()
    theta_latent_2 = (torch.randn((test_input.shape[0], 8), device=device)*0.01).requires_grad_()#theta_latent2_.clone().detach().requires_grad_()
    
    #optimizer = torch.optim.AdamW([theta_latent_1, theta_latent_2], lr=0.1, weight_decay=0, betas=(0.9, 0.99))
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    codebook = model.quantizer.codebook.detach().clone()
    
    #x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    #y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    vt1 = torch.zeros_like(theta_latent_1)
    vt2 = torch.zeros_like(theta_latent_2)

    eps=1e-8

    for step in range(epochs):
        t = random.randint(0, test_input.shape[1]-2)
        x_val = test_input[:, t]
        y_val = test_input[:, t+1]

        pred = simple_splitting(model, theta_latent_1, theta_latent_2, x_val, nt=1, dt=4/250, refinement_factor=1, splitting_method="strang")
        loss = relative_l2_error(pred, y_val)
        
        # Compute gradients using autograd
        grad_1 = torch.autograd.grad(loss, theta_latent_1, create_graph=True)[0]
        grad_2 = torch.autograd.grad(loss, theta_latent_2, create_graph=False)[0]
        
        # Riemannian gradient step
        riem_grad_1 = project_to_tangent(grad_1, theta_latent_1, codebook)
        riem_grad_2 = project_to_tangent(grad_2, theta_latent_2, codebook)

        eta = 0.1
        beta = 0.9
        
        vt1 =  beta*vt1 + (1-beta)*riem_grad_1**2
        vt2 =  beta*vt1 + (1-beta)*riem_grad_2**2
        
        theta_latent_1 = theta_latent_1 - eta * riem_grad_1/(torch.sqrt(vt1)+ eps)
        theta_latent_2 = theta_latent_2 - eta * riem_grad_2/(torch.sqrt(vt2)+ eps)
        
        # Retract to manifold (VQ quantization)
        theta_latent_1, _, _ = model.quantizer(theta_latent_1.unsqueeze(1))
        theta_latent_2, _, _ = model.quantizer(theta_latent_2.unsqueeze(1))
        theta_latent_1 = theta_latent_1.squeeze(1).detach().requires_grad_(True)
        theta_latent_2 = theta_latent_2.squeeze(1).detach().requires_grad_(True)

        print(step, loss.item(), (riem_grad_1**2).sum(), (riem_grad_2**2).sum())
        if step%100==0:
            with torch.no_grad():
                pred = simple_splitting(model, theta_latent_1, theta_latent_2, test_input[:, -1], nt=50, dt=4/250, refinement_factor=1, splitting_method="strang")
            test_error = loss_fn(pred, test_target).item()
            print(f"Grad selection, leading to error: {test_error:.6f}")

    return (theta_latent_1, theta_latent_2), pred, target

In [ ]:
def batch_cosine_aggregate(theta_batch, codebook, temperature=1.0):
    """
    Args:
        theta_batch: [batch_size, latent_dim] - continuous parameters
        codebook: [n_codes, latent_dim] - VQ codebook
        temperature: float - softmax temperature
    Returns:
        soft_embedding: [batch_size, latent_dim] - aggregated embeddings
        weights: [batch_size, n_codes] - attention weights
    """
    # Normalize for cosine similarity
    theta_norm = F.normalize(theta_batch, dim=1)
    codebook_norm = F.normalize(codebook, dim=1)
    
    # Cosine similarities: [batch_size, latent_dim] @ [latent_dim, n_codes]
    cosine_sims = torch.matmul(theta_norm, codebook_norm.T)
    
    # Softmax weights
    weights = F.softmax(cosine_sims / temperature, dim=1)
    
    # Aggregate: [batch_size, n_codes] @ [n_codes, latent_dim]
    soft_embedding = torch.matmul(weights, codebook)
    
    return soft_embedding, weights

In [ ]:
def gradient_selection_hybrid(model, theta_operators, test_input, test_target,
                             max_operators=5, min_improvement_threshold=5.0, 
                             epochs=200, lbfgs_steps=10, lr=0.1, refinement_factor=1):
    """Hybrid Adam + LBFGS approach."""
    if model is None or theta_operators is None:
        return [], {}
         
    print(f"Running hybrid gradient based operator selection...")
    print(f"Adam steps: {epochs}, LBFGS steps: {lbfgs_steps}")
         
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
         
    loss_fn = RelativeL2()
    theta_latent_1 = (torch.randn((test_input.shape[0], 3), device=device)*0.01).requires_grad_()
    theta_latent_2 = (torch.randn((test_input.shape[0], 3), device=device)*0.01).requires_grad_()
    
    # Phase 1: Adam warm-up
    print("Phase 1: Adam optimization...")
    optimizer_adam = torch.optim.AdamW([theta_latent_1, theta_latent_2], lr=lr, weight_decay=0, betas=(0.9, 0.99))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_adam, T_max=epochs)
         
    for step in range(epochs):
        t = random.randint(0, test_input.shape[1]-2)
        x_val = test_input[:, t]
        y_val = test_input[:, t+1]
                 
        pred = simple_splitting(model, theta_latent_1, theta_latent_2, x_val, nt=1, dt=4/250, refinement_factor=refinement_factor, splitting_method="strang")
        loss = relative_l2_error(pred, y_val)
                 
        optimizer_adam.zero_grad()
        loss.backward()
        optimizer_adam.step()
        scheduler.step()
        
        if step % 50 == 0:
            print(f"Adam step {step}, loss: {loss.item():.6f}")
    
    # Phase 2: LBFGS fine-tuning
    print("Phase 2: LBFGS fine-tuning...")
    optimizer_lbfgs = torch.optim.LBFGS(
        [theta_latent_1, theta_latent_2],
        lr=0.1,
        max_iter=20,        # more inner iterations
        history_size=10,    # more curvature info
        tolerance_grad=1e-7,
        tolerance_change=1e-9,
        line_search_fn="strong_wolfe"  # default, but explicit is better
    )

    
    current_t = 0
    def closure():
        optimizer_lbfgs.zero_grad()
        x_val = test_input[:, current_t]
        y_val = test_input[:, current_t+1:current_t+8+1]
        pred = simple_splitting(model, theta_latent_1, theta_latent_2, x_val, 
                              nt=8, dt=4/250, refinement_factor=refinement_factor, splitting_method="strang")
        loss = relative_l2_error(pred, y_val)
        loss.backward()
        return loss
    
    for step in range(lbfgs_steps):
        #current_t = random.randint(0, test_input.shape[1]-2)
        try:
            loss = optimizer_lbfgs.step(closure)
            print(f"LBFGS step {step}, loss: {loss.item():.6f}")
        except:
            print(f"LBFGS failed at step {step}, continuing...")
            continue
            
    # Final evaluation
    with torch.no_grad():
        pred = simple_splitting(model, theta_latent_1, theta_latent_2, test_input[:, -1], 
                              nt=50, dt=4/250, refinement_factor=refinement_factor, splitting_method="strang")
    test_error = loss_fn(pred, test_target).item()
    print(f"Final error: {test_error:.6f}")
            
    return (theta_latent_1, theta_latent_2), pred

In [ ]:
def gradient_selection(model, theta_operators, test_input, test_target, 
                             max_operators=5, min_improvement_threshold=5.0, epochs=100):
    """Greedy operator selection."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running gradient based operator selection...")
    print(f"Testing {theta_operators.shape[0]} operators, max length: {max_operators}")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()

    #theta_latent_1 = (torch.randn((test_input.shape[0], 8), device=device)*0.01).requires_grad_() #theta_latent1_.clone().detach().requires_grad_()
    #theta_latent_2 = (torch.randn((test_input.shape[0], 8), device=device)*0.01).requires_grad_()#theta_latent2_.clone().detach().requires_grad_()

    codebook = model.quantizer.codebook.detach()
    latent_dim = codebook.shape[1]

    batch_size = test_input.shape[0]
    theta_continuous_1 = torch.randn((batch_size, latent_dim), requires_grad=True, device=device)
    theta_continuous_2 = torch.randn((batch_size, latent_dim), requires_grad=True, device=device)
    
    optimizer = torch.optim.AdamW([theta_continuous_1, theta_continuous_2], lr=0.01, weight_decay=0) #betas=(0.9, 0.99))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    #x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    #y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    #eta= 10

    temperature = 1.0

    for step in range(epochs):
        t = random.randint(0, test_input.shape[1]-2)
        x_val = test_input[:, t]
        y_val = test_input[:, t+1]

        theta_latent_1, weights_1 = batch_cosine_aggregate(theta_continuous_1, codebook, temperature)
        theta_latent_2, weights_2 = batch_cosine_aggregate(theta_continuous_2, codebook, temperature)

        #temperature = temperature*0.95

        pred = simple_splitting(model, theta_latent_1, theta_latent_2, x_val, nt=1, dt=4/250, 
                       refinement_factor=1, splitting_method="strang")
        loss = relative_l2_error(pred, y_val)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        print(step, loss.item())
        if step%100==0:
            with torch.no_grad():
                pred_test = simple_splitting(model, theta_latent_1, theta_latent_2, test_input[:, -1], nt=50, dt=4/250, refinement_factor=1, splitting_method="strang")
            test_error = loss_fn(pred_test, test_target).item()
            print(f"Grad selection, leading to error: {test_error:.6f}")

    return (theta_latent_1, theta_latent_2, weights_1, weights_2), pred_test

In [ ]:
TRAINING_FILES_EASY = {}
for key in TRAINING_FILES.keys():
    if key != 'DISP':
        TRAINING_FILES_EASY[key] = TRAINING_FILES[key]

In [ ]:
#train_files = [TRAINING_FILES[key] for key in TRAINING_FILES.keys()]

theta_operators, theta_latent_operators, operator_metadata = encode_operators_from_training_data(model, TRAINING_FILES, num_operators=1024, n_trajectories_per_operator=1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# Your theta encodings from the neural network (replace with actual data)
theta = theta_latent_operators# Shape (n, 3) - replace with your actual theta

# Your metadata template with true parameters
template_info = operator_metadata

# Extract theta coordinates (neural network encodings)
theta_x = theta[:, 0]
theta_y = theta[:, 1] 
theta_z = theta[:, 2]

# Extract true parameters from metadata (handle PyTorch tensors)
true_alpha = [info['alpha'].item() if hasattr(info['alpha'], 'item') else info['alpha'] for info in template_info]
true_beta = [info['beta'].item() if hasattr(info['beta'], 'item') else info['beta'] for info in template_info]
true_gamma = [info['gamma'].item() if hasattr(info['gamma'], 'item') else info['gamma'] for info in template_info]

# Color mapping for equation types
color_map = {'EULER': '#FF6B6B', 'HEAT': '#4ECDC4', 'DISP': '#45B7D1', 'OTHER': '#96CEB4'}
colors = [color_map.get(info['equation_type'], color_map['OTHER']) for info in template_info]

# Create the 3D plot
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot points by equation type for proper legend
equation_types = list(set(info['equation_type'] for info in template_info))

for eq_type in equation_types:
    # Get indices for this equation type
    indices = [i for i, info in enumerate(template_info) if info['equation_type'] == eq_type]
    
    if indices:
        ax.scatter(theta_x[indices], theta_y[indices], theta_z[indices], 
                  c=color_map[eq_type], label=eq_type, s=60, alpha=0.7, 
                  edgecolors='black', linewidth=0.5)

# Only label a few representative points to avoid clutter
sample_indices = [0, len(template_info)//3, 2*len(template_info)//3, -1]  # Sample a few points
for i in sample_indices:
    if i < len(template_info):
        info = template_info[i]
        alpha_val = info['alpha'].item() if hasattr(info['alpha'], 'item') else info['alpha']
        beta_val = info['beta'].item() if hasattr(info['beta'], 'item') else info['beta']
        gamma_val = info['gamma'].item() if hasattr(info['gamma'], 'item') else info['gamma']
        
        # Only show the most relevant parameter (non-zero one)
        if alpha_val != 0:
            param_str = f"α={alpha_val:.3f}"
        elif beta_val != 0:
            param_str = f"β={beta_val:.3f}"
        elif gamma_val != 0:
            param_str = f"γ={gamma_val:.3f}"
        else:
            param_str = "zeros"
            
        label = f"{info['equation_type']}\n{param_str}"
        ax.text(theta_x[i], theta_y[i], theta_z[i], f"  {label}", 
                fontsize=9, ha='left', va='bottom',
                bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.8, edgecolor='gray'))

# Customize the plot
ax.set_xlabel('θ₁ (Neural Encoding Dim 1)', fontsize=12, labelpad=10)
ax.set_ylabel('θ₂ (Neural Encoding Dim 2)', fontsize=12, labelpad=10)
ax.set_zlabel('θ₃ (Neural Encoding Dim 3)', fontsize=12, labelpad=10)
ax.set_title('Neural Network Encodings (θ) vs True Parameters (α,β,γ)', fontsize=14, pad=20)

# Add legend
ax.legend(loc='upper left', bbox_to_anchor=(0.02, 0.98), fontsize=10)

# Improve visualization
ax.grid(True, alpha=0.3)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

# Set viewing angle for better visualization
ax.view_init(elev=20, azim=45)

# Add text box with summary statistics
textstr = f"""Encoding Statistics:
θ₁: [{theta_x.min():.3f}, {theta_x.max():.3f}]
θ₂: [{theta_y.min():.3f}, {theta_y.max():.3f}]
θ₃: [{theta_z.min():.3f}, {theta_z.max():.3f}]

True Parameter Ranges:
α: [{min(true_alpha):.3f}, {max(true_alpha):.3f}]
β: [{min(true_beta):.3f}, {max(true_beta):.3f}]
γ: [{min(true_gamma):.3f}, {max(true_gamma):.3f}]"""

props = dict(boxstyle='round', facecolor='lightgray', alpha=0.8)
ax.text2D(0.02, 0.02, textstr, transform=ax.transAxes, fontsize=9,
          verticalalignment='bottom', bbox=props)

plt.tight_layout()
plt.show()

# Optional: Create a 2D correlation plot to see how encodings relate to true parameters
fig2, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot theta vs true alpha
for eq_type in equation_types:
    indices = [i for i, info in enumerate(template_info) if info['equation_type'] == eq_type]
    if indices:
        axes[0].scatter([true_alpha[i] for i in indices], [theta_x[i] for i in indices], 
                       c=color_map[eq_type], label=eq_type, s=80, alpha=0.7)

axes[0].set_xlabel('True α')
axes[0].set_ylabel('θ₁ (Encoding)')
axes[0].set_title('Neural Encoding vs True Alpha')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot theta vs true beta  
for eq_type in equation_types:
    indices = [i for i, info in enumerate(template_info) if info['equation_type'] == eq_type]
    if indices:
        axes[1].scatter([true_beta[i] for i in indices], [theta_y[i] for i in indices], 
                       c=color_map[eq_type], label=eq_type, s=80, alpha=0.7)

axes[1].set_xlabel('True β')
axes[1].set_ylabel('θ₂ (Encoding)')
axes[1].set_title('Neural Encoding vs True Beta')
axes[1].grid(True, alpha=0.3)

# Plot theta vs true gamma
for eq_type in equation_types:
    indices = [i for i, info in enumerate(template_info) if info['equation_type'] == eq_type]
    if indices:
        axes[2].scatter([true_gamma[i] for i in indices], [theta_z[i] for i in indices], 
                       c=color_map[eq_type], label=eq_type, s=80, alpha=0.7)

axes[2].set_xlabel('True γ')
axes[2].set_ylabel('θ₃ (Encoding)')
axes[2].set_title('Neural Encoding vs True Gamma')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Plots created!")
print(f"Neural encoding shape: {theta.shape}")
print("This shows how your neural network has learned to encode the true physical parameters.")

In [ ]:
encoded_operators = theta_operators, theta_latent_operators, operator_metadata 

In [ ]:
alphas = np.array([m['alpha'] for m in operator_metadata]).squeeze()
betas = np.array([m['beta'] for m in operator_metadata]).squeeze()
gammas = np.array([m['gamma'] for m in operator_metadata]).squeeze()
alphas.sort()
betas.sort()
gammas.sort()

In [ ]:
#tt = np.unique(theta_latent_operators, axis=0)
#unique_rows, indices, inverse = np.unique(
#    theta_latent_operators, axis=0, return_index=True, return_inverse=True
#)

In [ ]:
#encoded_operators = theta_operators[indices], unique_rows, [operator_metadata[i] for i in indices]

In [ ]:
theta_latent_operators.std()

In [ ]:
# Usage:
N_OUTPUT_FRAMES=2
results = test_all_ood_datasets(['E_BG'], epochs=1000, num_operators=2, lr=0.1, refinement_factor=1)

In [ ]:
# HEAT + BURGERS does not work it seems

In [ ]:
# try do do nearest neighbor parameter by parameter and see if it works

In [ ]:
target = results['E_BG']['target_grad']

In [ ]:
u_composition = results['E_BG']['pred_grad']

In [ ]:
((u_composition.cpu().squeeze()-target.cpu().squeeze())**2).mean(-1).mean(-1)

In [ ]:
# with manifold loss
# BG: 0.008 
# HE: 0.025
# ED: 0.055
# ALL: 0.03

# refinement_factor=1
# BG: 0.0181
# ED: 0.063 
# HE: 0.057

# refinement_factor=5
# BG:
# ED: 0.073

In [ ]:
theta_latent_operators

In [ ]:
theta_latent_operators
theta_1 = results["E_BG"]['theta_latent'][0]
theta_2 = results["E_BG"]['theta_latent'][1]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def plot_thetas(theta_latent_operators, theta_1, theta_2):
    """Simple plot to see where theta parameters end up."""
    
    # Convert to numpy
    if torch.is_tensor(theta_latent_operators):
        ops = theta_latent_operators.detach().cpu().numpy()
    else:
        ops = theta_latent_operators
        
    if torch.is_tensor(theta_1):
        t1 = theta_1.detach().cpu().numpy()
    else:
        t1 = theta_1
        
    if torch.is_tensor(theta_2):
        t2 = theta_2.detach().cpu().numpy()
    else:
        t2 = theta_2
    
    # Simple scatter plot
    plt.figure(figsize=(10, 6))
    
    # Plot dim 0 vs dim 1
    plt.subplot(1, 2, 1)
    plt.scatter(ops[:, 0], ops[:, 1], c='gray', alpha=0.5, label='Reference operators')
    plt.scatter(t1[:, 0], t1[:, 1], c='red', s=1, label='θ₁')
    plt.scatter(t2[:, 0], t2[:, 1], c='blue', s=1, label='θ₂')
    plt.xlabel('Dimension 0')
    plt.ylabel('Dimension 1')
    plt.legend()
    plt.title('Theta positions')
    
    # Plot dim 0 vs dim 2
    plt.subplot(1, 2, 2)
    plt.scatter(ops[:, 0], ops[:, 2], c='gray', alpha=0.5, label='Reference operators')
    plt.scatter(t1[:, 0], t1[:, 2], c='red', s=1, label='θ₁')
    plt.scatter(t2[:, 0], t2[:, 2], c='blue', s=1, label='θ₂')
    plt.xlabel('Dimension 0')
    plt.ylabel('Dimension 2')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Usage:
plot_thetas(theta_latent_operators, theta_1, theta_2)

In [ ]:
u_composition.shape

In [ ]:
idx=112

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(u_composition[idx].squeeze(1).cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze(1).cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze(1).cpu().detach()[t])

## Operator Encoding

## Operator Selection Methods

In [ ]:
def simple_splitting(model, theta_latent_1, theta_latent_2, x, nt=1, dt=4/250, refinement_factor=5, splitting_method="strang"):
    
    state_labels = torch.tensor([0], device=x.device)
    #state_labels = 0
    dim = 1
    
    small_dt = dt/refinement_factor
    small_dt_half = small_dt/2

    theta_1 = model.decode_theta(theta_latent_1, dim)
    theta_2 = model.decode_theta(theta_latent_2, dim)

    pred = x
    trajectory_pred = []
    for t_idx in range(nt):
        for _ in range(refinement_factor):
            if splitting_method == 'strang':
                pred, metadata = model.solve_ode(pred, theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt_half, dt=small_dt_half)
                pred, metadata = model.solve_ode(pred[:, -1], theta_2, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred, metadata = model.solve_ode(pred[:, -1], theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt_half, dt=small_dt_half)
                pred = pred[:, -1]

            else:  # lie
                # Lie splitting: full step op1, full step op2
                pred, metadata = model.solve_ode(pred, theta_1, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred, metadata = model.solve_ode(pred[:, -1], theta_2, state_labels, dim, n_future_steps=1, integration_time=small_dt, dt=small_dt)
                pred = pred[:, -1]

        # Store prediction at each time step
        trajectory_pred.append(pred)

    # Convert trajectory to numpy array: (n_t+1, batch_size, 2, n_x, n_y)
    trajectory_pred = torch.cat(trajectory_pred, axis=1)
    return trajectory_pred


In [ ]:
import torch
import numpy as np
from typing import List, Tuple
from torch import nn

class RelativeL2(nn.Module):
    def forward(self, x, y):
        x = rearrange(x, "b ... -> b (...)")
        y = rearrange(y, "b ... -> b (...)")
        diff_norms = torch.linalg.norm(x - y, ord=2, dim=-1)
        y_norms = torch.linalg.norm(y, ord=2, dim=-1)
        return (diff_norms / y_norms)


class DiscreteOperatorEvolution:
    def __init__(self, codebook, population_size=50, elite_size=10, mutation_rate=0.2):
        self.codebook = codebook  # [n_codes, latent_dim]
        self.n_codes = len(codebook)
        self.population_size = population_size
        self.elite_size = elite_size
        self.mutation_rate = mutation_rate
        
        # Population: each individual is (op1_index, op2_index)
        self.population = self._initialize_population()
        self.fitness_history = []
        self.relative_l2_error = RelativeL2()
        
    def _initialize_population(self):
        """Initialize population of discrete operator index pairs"""
        population = []
        for _ in range(self.population_size):
            op1_idx = np.random.randint(0, self.n_codes)
            op2_idx = np.random.randint(0, self.n_codes)
            population.append((op1_idx, op2_idx))
        return population
    
    def _decode_individual(self, individual):
        """Convert indices to actual operator embeddings"""
        theta_1 = self.codebook[individual[:, 0]]
        theta_2 = self.codebook[individual[:, 1]]
        return theta_1, theta_2
    
    def evaluate_fitness(self, individual, model, x_val, y_val):
        """Evaluate fitness of an individual"""
        #theta_1, theta_2 = self._decode_individual(individual)
        theta_1, theta_2 = self._decode_individual(individual)
        
        # Expand for batch processing if needed
        #if x_val.dim() > 1:
            #theta_1 = theta_1.unsqueeze(0).expand(x_val.shape[0], -1)
            #theta_2 = theta_2.unsqueeze(0).expand(x_val.shape[0], -1)
        
        with torch.no_grad():
            x_val = x_val.repeat(theta_1.shape[0], 1, 1)
            pred = simple_splitting(model, theta_1, theta_2, x_val, nt=1, dt=4/250, 
                                  refinement_factor=5, splitting_method="strang")
            #print(pred)
            loss = self.relative_l2_error(pred, y_val).cpu().numpy()
        return -loss
    
    def crossover(self, parent1, parent2):
        """Single-point crossover for discrete indices"""
        op1_p1, op2_p1 = parent1
        op1_p2, op2_p2 = parent2
        
        # Simple crossover: swap one operator
        if np.random.rand() < 0.5:
            child1 = (op1_p1, op2_p2)
            child2 = (op1_p2, op2_p1)
        else:
            child1 = (op1_p2, op2_p1)
            child2 = (op1_p1, op2_p2)
        
        return child1, child2
    
    def mutate(self, individual):
        """Discrete mutation: randomly change operator indices"""
        op1_idx, op2_idx = individual
        
        # Mutate operator 1
        if np.random.rand() < self.mutation_rate:
            op1_idx = np.random.randint(0, self.n_codes)
        
        # Mutate operator 2
        if np.random.rand() < self.mutation_rate:
            op2_idx = np.random.randint(0, self.n_codes)
        
        return (op1_idx, op2_idx)
    
    def evolve_generation(self, model, x_val, y_val):
        """Evolve one generation"""
        # Evaluate fitness
        fitness_scores = []
        #for individual in self.population:
        pop = torch.tensor(self.population)
        fitness_scores = self.evaluate_fitness(pop, model, x_val, y_val)
        #fitness_scores.append(fitness)
        
        # Sort by fitness
        sorted_indices = np.argsort(fitness_scores)[::-1]  # descending
        
        # Select elite
        elite = [self.population[i] for i in sorted_indices[:self.elite_size]]
        
        # Generate new population
        new_population = elite.copy()
        
        while len(new_population) < self.population_size:
            # Tournament selection
            parent1 = self._tournament_select(fitness_scores)
            parent2 = self._tournament_select(fitness_scores)
            
            # Crossover
            child1, child2 = self.crossover(parent1, parent2)
            
            # Mutation
            child1 = self.mutate(child1)
            child2 = self.mutate(child2)
            
            new_population.extend([child1, child2])
        
        self.population = new_population[:self.population_size]
        self.fitness_history.append(max(fitness_scores))
        
        return max(fitness_scores), elite[0]  # best fitness and best individual
    
    def _tournament_select(self, fitness_scores, tournament_size=3):
        """Tournament selection"""
        indices = np.random.choice(len(fitness_scores), tournament_size, replace=False)
        winner_idx = indices[np.argmax([fitness_scores[i] for i in indices])]
        return self.population[winner_idx]


In [ ]:
N_OUTPUT_FRAMES=50
dataloader = load_ood_dataset("E_ED")

In [ ]:
for batch in dataloader:
    test_input = batch["input"]
    test_target = batch["target"]
    break

In [ ]:
codebook = model.quantizer.codebook.clone()
discrete_evo = DiscreteOperatorEvolution(codebook, population_size=128, elite_size=16, mutation_rate=0.2)
idx = random.randint(0, test_input.shape[0])
x = test_input[idx, -1].unsqueeze(0).cuda()
y = test_target[idx].unsqueeze(0).cuda()
epochs=100
for generation in range(epochs):
    t = random.randint(0, test_input.shape[1]-2)
    x_val = test_input[idx, t].cuda().unsqueeze(0)
    y_val = test_input[idx, t+1].cuda().unsqueeze(0)
    
    # Evolve population
    best_fitness, best_individual = discrete_evo.evolve_generation(model, x_val, y_val)
    
    # Get best operators for this generation
    theta_latent_1, theta_latent_2 = discrete_evo._decode_individual(torch.tensor(best_individual).unsqueeze(0))
    
    if generation % 20 == 0:
        print(f"Generation {generation}: Best fitness = {best_fitness:.6f}")
        print(f"Best operators: {best_individual}")
        #print('theta_latent_1', theta_latent_1.shape, theta_latent_1.dtype)

    if generation == epochs-1:
        with torch.no_grad():
            pred = simple_splitting(model, theta_latent_1, theta_latent_2, x, nt=N_OUTPUT_FRAMES, dt=4/250, refinement_factor=5, splitting_method="strang")
        error = discrete_evo.relative_l2_error(pred, y)
        print(f"Extrapolation error: {error.mean()}")

        

In [ ]:
lit_model

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(pred.squeeze().cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(y.squeeze().cpu().detach()[t])